# indicators_002

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (53).ipynb`

| Field | Value |
|---|---|
| Section | `indicators` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `daily` |
| Code cells | 6 |
| Detected functions | compute_vwap, compute_obv, compute_breadth, compute_mfi, compute_atr, compute_hv |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd

data = pd.read_parquet('/content/drive/MyDrive/ipynb/Quarter/HDFCBANK_NSE.parquet')
data

In [ ]:
import pandas as pd

# Ensure Date_Time is in datetime format
data['Date_Time'] = pd.to_datetime(data['Date_Time'])

# Filter out rows where time is greater than 15:30
data = data[data['Date_Time'].dt.time <= pd.to_datetime('15:30').time()]

# Reset index after filtering
data = data.reset_index(drop=True)

# Display the first few rows
print(data.head())


In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# Ensure Date_Time is in datetime format
data['Date_Time'] = pd.to_datetime(data['Date_Time'])

# Extract Date for Grouping
data['Date'] = data['Date_Time'].dt.date

# Sort data by Ticker and Date_Time for correct calculations
data = data.sort_values(by=['Ticker', 'Date_Time']).reset_index(drop=True)

tqdm.pandas(desc="Processing Indicators")

### Liquidity Indicators ###
print("Computing Liquidity Indicators...")
data['Bid-Ask Spread'] = data['SellPrice'] - data['BuyPrice']
data['Liquidity Imbalance'] = data['BuyQty'] - data['SellQty']
data['Order Book Depth'] = (data['BuyQty'] + data['SellQty']) / 2
data['Slippage'] = np.abs(data['LTP'] - data['BuyPrice']) / data['BuyPrice']

### Grouping by Day for VWAP & OBV ###
print("Computing VWAP & OBV per Day...")

def compute_vwap(df):
    return (df['LTP'] * df['LTQ']).cumsum() / df['LTQ'].cumsum()

def compute_obv(df):
    return (np.sign(df['LTP'].diff()) * df['LTQ']).fillna(0).cumsum()

# Compute VWAP and OBV per Day & Ticker
data['VWAP'] = data.groupby(['Ticker', 'Date']).progress_apply(compute_vwap).reset_index(level=[0,1], drop=True)
data['OBV'] = data.groupby(['Ticker', 'Date']).progress_apply(compute_obv).reset_index(level=[0,1], drop=True)

### Momentum Indicators ###
print("Computing Momentum Indicators...")
window_rsi = 14
delta = data['LTP'].diff()
gain = np.where(delta > 0, delta, 0)
loss = np.where(delta < 0, -delta, 0)

data['RSI'] = (
    100 - (100 / (1 + (pd.Series(gain).rolling(window=window_rsi).mean() /
                       (pd.Series(loss).rolling(window=window_rsi).mean() + 1e-10))))
)

data['ROC'] = data['LTP'].pct_change(periods=5) * 100
data['Momentum'] = data['LTP'] - data['LTP'].shift(5)

### Market Breadth (Daily Advancing/Declining) ###
print("Computing Market Breadth Indicators...")

def compute_breadth(df):
    df['Advancing'] = (df['LTP'].diff() > 0).astype(int).cumsum()
    df['Declining'] = (df['LTP'].diff() < 0).astype(int).cumsum()
    return df[['Advancing', 'Declining']]

breadth = data.groupby(['Ticker', 'Date']).progress_apply(compute_breadth)
data[['Advancing', 'Declining']] = breadth.reset_index(level=[0,1], drop=True)

### Mean Reversion Indicators ###
print("Computing Mean Reversion Indicators...")
window = 20
mean = data['LTP'].rolling(window=window).mean()
std_dev = data['LTP'].rolling(window=window).std()
data['Z-Score'] = (data['LTP'] - mean) / std_dev
data['Bollinger Upper'] = mean + (2 * std_dev)
data['Bollinger Lower'] = mean - (2 * std_dev)

### Money Flow Indicators (Reset Daily) ###
print("Computing Money Flow Indicators...")

def compute_mfi(df):
    df['Raw Money Flow'] = df['LTP'] * df['LTQ']
    df['Positive Flow'] = np.where(df['LTP'].diff() > 0, df['Raw Money Flow'], 0)
    df['Negative Flow'] = np.where(df['LTP'].diff() < 0, df['Raw Money Flow'], 0)
    df['Money Flow Ratio'] = df['Positive Flow'].rolling(14).sum() / (df['Negative Flow'].rolling(14).sum() + 1e-10)
    df['MFI'] = 100 - (100 / (1 + df['Money Flow Ratio']))
    return df['MFI']

data['MFI'] = data.groupby(['Ticker', 'Date']).progress_apply(compute_mfi).reset_index(level=[0,1], drop=True)

### Volatility Indicators (Daily) ###
print("Computing Volatility Indicators...")

def compute_atr(df):
    df['ATR'] = df['LTP'].rolling(14).apply(lambda x: np.max(x) - np.min(x))
    return df['ATR']

def compute_hv(df):
    df['Historical Volatility'] = df['LTP'].pct_change().rolling(20).std() * np.sqrt(252)
    return df['Historical Volatility']

data['ATR'] = data.groupby(['Ticker', 'Date']).progress_apply(compute_atr).reset_index(level=[0,1], drop=True)
data['Historical Volatility'] = data.groupby(['Ticker', 'Date']).progress_apply(compute_hv).reset_index(level=[0,1], drop=True)

# Drop NaN values created by rolling calculations
data.dropna(inplace=True)

# Display the first few rows
print(data.head())


In [ ]:
import pandas as pd
import numpy as np

# Assuming 'data' is already loaded with columns: ['date', 'open', 'high', 'low', 'close', 'volume']

# Liquidity Imbalance (Difference in volumes of buyers and sellers)
data['Liquidity Imbalance'] = data['volume'].diff()

# Order Book Depth (Using volume as a proxy for depth)
data['Order Book Depth'] = data.groupby('date')['volume'].cumsum()

# Slippage (Difference between close and open prices)
data['Slippage'] = data['close'] - data['open']

# VWAP (Volume Weighted Average Price) grouped by day
data['VWAP'] = (data['close'] * data['volume']).groupby(data['date']).cumsum() / (data['volume'].groupby(data['date']).cumsum() + 1e-9)

# OBV (On-Balance Volume)
data['OBV'] = data.groupby('date')['close'].diff().apply(np.sign) * data['volume']
data['OBV'] = data.groupby('date')['OBV'].cumsum()

# RSI (Relative Strength Index)
window = 14
delta = data['close'].diff()
gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)
rs = gain.ewm(span=window, adjust=False).mean() / loss.ewm(span=window, adjust=False).mean()
data['RSI'] = 100 - (100 / (1 + rs))

# ROC (Rate of Change)
data['ROC'] = data['close'].pct_change(periods=window) * 100

# Momentum (Difference in closing price over time)
data['Momentum'] = data['close'].diff(periods=window)

# Advancing and Declining Stocks
data['Advancing'] = (data['close'] > data['close'].shift(1)).astype(int)
data['Declining'] = (data['close'] < data['close'].shift(1)).astype(int)

# Z-Score
data['Z-Score'] = (data['close'] - data['close'].rolling(window).mean()) / data['close'].rolling(window).std()

# Bollinger Bands
data['Bollinger Mean'] = data.groupby('date')['close'].transform(lambda x: x.rolling(window).mean())
data['Bollinger Std'] = data.groupby('date')['close'].transform(lambda x: x.rolling(window).std())
data['Bollinger Upper'] = data['Bollinger Mean'] + (2 * data['Bollinger Std'])
data['Bollinger Lower'] = data['Bollinger Mean'] - (2 * data['Bollinger Std'])

# MFI (Money Flow Index)
typical_price = (data['high'] + data['low'] + data['close']) / 3
money_flow = typical_price * data['volume']
positive_flow = money_flow.where(typical_price > typical_price.shift(1), 0)
negative_flow = money_flow.where(typical_price < typical_price.shift(1), 0)
mf_ratio = positive_flow.rolling(window).sum() / (negative_flow.rolling(window).sum() + 1e-9)
data['MFI'] = 100 - (100 / (1 + mf_ratio))

# ATR (Average True Range)
tr = np.maximum(data['high'] - data['low'],
                np.maximum(abs(data['high'] - data['close'].shift(1)),
                           abs(data['low'] - data['close'].shift(1))))
data['ATR'] = tr.ewm(span=window, adjust=False).mean()

# ADX (Average Directional Index)
plus_dm = np.where((data['high'].diff() > data['low'].diff()) & (data['high'].diff() > 0), data['high'].diff(), 0)
minus_dm = np.where((data['low'].diff() > data['high'].diff()) & (data['low'].diff() > 0), data['low'].diff(), 0)
tr = np.maximum(data['high'] - data['low'],
                np.maximum(abs(data['high'] - data['close'].shift(1)),
                           abs(data['low'] - data['close'].shift(1))))
plus_di = 100 * (pd.Series(plus_dm, index=data.index).ewm(span=window, adjust=False).mean() / pd.Series(tr, index=data.index).ewm(span=window, adjust=False).mean())
minus_di = 100 * (pd.Series(minus_dm, index=data.index).ewm(span=window, adjust=False).mean() / pd.Series(tr, index=data.index).ewm(span=window, adjust=False).mean())
dx = 100 * (abs(plus_di - minus_di) / (plus_di + minus_di + 1e-9))
data['ADX'] = dx.ewm(span=window, adjust=False).mean()

# Historical Volatility
data['Historical Volatility'] = data['close'].pct_change().rolling(window).std() * np.sqrt(252)

# Drop NaN values
data.dropna(inplace=True)

print(data.head())


In [ ]:
data

In [ ]:
data.to_parquet('/content/drive/MyDrive/data_w_ind/HDFCBANK_IND.parquet')